# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AroojSaleem-995/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Dataset and Setup

This notebook builds a transparent baseline for the Refresh / Content
Opportunity lane.

The dataset contains 30,000 rows and 44 columns. The baseline uses current
snapshot signals that are relevant to the rule:

- `impressions_90d` — search visibility / volume
- `ctr` — observed click-through rate
- `avg_position` — average search position

The notebook first checks these signals, then builds one hand-written score,
one reason code, and one action label.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [16]:
signal_cols = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

print(df[signal_cols].describe())

       impressions_90d           ctr  avg_position
count     30000.000000  30000.000000   30000.00000
mean       5200.366300      0.510733      16.34238
std       16838.019547      3.279162      15.21679
min           1.000000      0.000000       0.00000
25%          81.000000      0.000000       6.20000
50%         731.000000      0.070000      10.80000
75%        3615.250000      0.290000      22.30000
max      517715.000000    100.000000     245.00000


In [17]:
print(df[signal_cols].isna().sum())

impressions_90d    0
ctr                0
avg_position       0
dtype: int64


In [18]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_table = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean")
      )
      .reset_index()
)

position_table

,position_bucket,n,avg_ctr,avg_impressions,avg_clicks
0,1-3,2346,1.472869,3223.757033,15.792413
1,4-10,11842,0.651045,7546.142543,26.340821
2,11-20,7273,0.323443,3137.629589,10.937990
3,21+,8539,0.211333,4247.178241,6.369715


## Signal 1: CTR vs. Search Position

### Why this signal was checked

The baseline uses the relationship between click-through rate (CTR) and
average search position. This signal is linked to FlyRank's CTR-vs-position
logic used in the CTR-fix flag.

The idea is that pages ranking higher in search results should generally
receive a higher CTR, while pages ranking lower should generally receive
a lower CTR. Checking this relationship helps determine whether CTR is a
meaningful signal for identifying potential content opportunities.

### Bucket analysis

The pages were divided into four average-position buckets:

- **1–3:** 2,346 pages, average CTR = 1.4729
- **4–10:** 11,842 pages, average CTR = 0.6510
- **11–20:** 7,273 pages, average CTR = 0.3234
- **21+:** 8,539 pages, average CTR = 0.2113

The bucket table shows a clear and consistent decrease in average CTR as
search position becomes worse. Pages ranking in positions 1–3 have the
highest average CTR, while pages ranking 21+ have the lowest average CTR.

### Verdict: CONFIRMED

The CTR-vs-position signal is **CONFIRMED**. The observed data supports the
expected relationship between search position and CTR: higher-ranking pages
receive substantially higher CTR than lower-ranking pages.

This makes CTR relative to search position a reasonable signal for the
baseline. However, CTR alone does not prove that content needs refreshing;
a low CTR can also result from SERP features, search intent, titles,
descriptions, or other factors. Therefore, this signal should be combined
with search-volume information rather than used by itself.

In [19]:
df["impression_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)

impression_table = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          n=("impressions_90d", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

impression_table

,impression_bucket,n,avg_impressions,avg_clicks,avg_ctr,avg_position
0,"(0.999, 81.0]",7503,19.668533,0.135812,1.265650,13.279328
1,"(81.0, 731.0]",7499,334.827310,0.710628,0.237681,21.491306
2,"(731.0, 3615.25]",7498,1785.258069,4.337290,0.228640,17.112137
3,"(3615.25, 517715.0]",7500,18662.224133,59.206800,0.310549,13.488867


## Signal 2: Search Volume / Impressions

### Why this signal was checked

The second signal is `impressions_90d`, which represents how often pages
were shown in search results during the 90-day observation window.

This signal is relevant to the content-opportunity baseline because a page
with substantial search visibility has more potential value from improvement
than a page that receives very little search exposure. It is also related to
the volume-based quick-win logic discussed in the FlyRank session.

### Bucket analysis

The pages were divided into four equal-sized impression buckets:

- **Lowest impressions:** 7,503 pages, average impressions = 19.67,
  average clicks = 0.14
- **Low impressions:** 7,499 pages, average impressions = 334.83,
  average clicks = 0.71
- **High impressions:** 7,498 pages, average impressions = 1,785.26,
  average clicks = 4.33
- **Highest impressions:** 7,500 pages, average impressions = 18,662.22,
  average clicks = 59.21

The results show a strong increase in average clicks as search impressions
increase. The highest-impression bucket receives substantially more clicks
than the lower-impression buckets.

### Verdict: CONFIRMED

The search-volume signal is **CONFIRMED**. Pages with greater search
visibility tend to generate substantially more clicks, indicating that
`impressions_90d` is a useful signal for prioritizing pages with greater
potential search opportunity.

However, impressions alone do not mean that a page needs refreshing. A page
can have high impressions and already perform well. Therefore, impressions
should be combined with another signal, such as the CTR-vs-position gap,
before assigning a refresh action.

In [20]:
expected_ctr_by_position = (
    df.groupby("position_bucket", observed=False)["ctr"]
      .median()
)

df["expected_ctr"] = df["position_bucket"].map(
    expected_ctr_by_position
)

df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr"]
)

df[[
    "avg_position",
    "ctr",
    "expected_ctr",
    "ctr_gap"
]].head()

,avg_position,ctr,expected_ctr,ctr_gap
0,10.6,0.76,0.10,-0.66
1,20.3,0.05,0.00,-0.05
2,36.5,0.09,0.00,-0.09
3,6.2,0.49,0.16,-0.33
4,44.0,0.13,0.00,-0.13


### CTR-gap limitation

Some position buckets have a median CTR of 0.00 in this dataset. As a
result, several pages receive the same non-positive CTR gap and therefore
have limited separation in the CTR opportunity score.

This is acceptable for the baseline because the goal is a simple,
transparent benchmark rather than a fully optimized scoring model. It is
also a limitation that a later model can address.

In [35]:
df["volume_score"] = (
    df["impressions_90d"].rank(pct=True)
)

df["volume_score"] = (
    df["impressions_90d"].rank(pct=True)
)

positive_gap = df["ctr_gap"].clip(lower=0)

df["ctr_opportunity_score"] = np.where(
    positive_gap > 0,
    positive_gap.rank(pct=True),
    0
)
df[[
    "volume_score",
    "ctr_opportunity_score"
]].describe()

,volume_score,ctr_opportunity_score
count,30000.000000,30000.000000
mean,0.500017,0.264250
std,0.288669,0.394094
min,0.017933,0.000000
25%,0.249817,0.000000
50%,0.499933,0.000000
75%,0.750012,0.747533
max,1.000000,0.932583


## 4. Baseline Action Score

The baseline combines the two signals that were validated in Section 1:

1. **Search volume score** — based on the percentile rank of `impressions_90d`.
   Pages with greater search visibility receive a higher volume score.

2. **CTR opportunity score** — based on the percentile rank of the positive
   CTR gap relative to the typical CTR for the page's position bucket.
   Pages with a larger positive CTR gap receive a higher opportunity score.

The final baseline score is a weighted combination of these two signals:

**Baseline Score = 0.60 × Volume Score + 0.40 × CTR Opportunity Score**

Search volume receives 60% of the weight because a content improvement is
more valuable to prioritize when the page has meaningful existing search
visibility. The CTR opportunity receives 40% because it identifies pages
that may be underperforming relative to their search position.

This is a transparent, hand-written baseline rather than a learned machine
learning model. The purpose is to create a simple benchmark that the Week-5
model can later be compared against.

The score uses only current 90-day snapshot signals and derived features.
No future-window performance or outcome/label information is used.

In [23]:
df["baseline_score"] = (
    0.6 * df["volume_score"]
    + 0.4 * df["ctr_opportunity_score"]
)

In [24]:
df["reason_code"] = np.where(
    df["ctr_opportunity_score"] >= df["volume_score"],
    "CTR_GAP",
    "HIGH_VOLUME"
)

### Action Rule

Pages at or above the 75th percentile of the baseline score are assigned
`REFRESH_REVIEW`. All other pages are assigned `MONITOR`.

This creates a clear review queue containing the highest-scoring 25% of
pages. The action is a prioritization recommendation, not proof that the
content must be refreshed.

In [25]:
action_threshold = df["baseline_score"].quantile(0.75)

df["action"] = np.where(
    df["baseline_score"] >= action_threshold,
    "REFRESH_REVIEW",
    "MONITOR"
)

In [26]:
ranked = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1

In [27]:
ranked[[
    "rank",
    "baseline_score",
    "action",
    "reason_code",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ctr_gap"
]].head(20)

,rank,baseline_score,action,reason_code,impressions_90d,ctr,avg_position,ctr_gap
0,1,0.972473,REFRESH_REVIEW,HIGH_VOLUME,208678,0.00,9.7,0.16
1,2,0.944387,REFRESH_REVIEW,HIGH_VOLUME,140079,0.01,7.6,0.15
2,3,0.944253,REFRESH_REVIEW,HIGH_VOLUME,223271,0.03,7.8,0.13
3,4,0.943187,REFRESH_REVIEW,HIGH_VOLUME,119217,0.02,7.0,0.14
4,5,0.943147,REFRESH_REVIEW,HIGH_VOLUME,112434,0.01,7.2,0.15
5,6,0.943073,REFRESH_REVIEW,HIGH_VOLUME,134055,0.03,7.5,0.13
6,7,0.942433,REFRESH_REVIEW,HIGH_VOLUME,123469,0.03,8.0,0.13
7,8,0.942173,REFRESH_REVIEW,HIGH_VOLUME,22456,0.00,6.6,0.16
8,9,0.942140,REFRESH_REVIEW,HIGH_VOLUME,295097,0.05,7.3,0.11
9,10,0.941253,REFRESH_REVIEW,HIGH_VOLUME,99013,0.03,6.4,0.13


### Ranked Queue and Output

The pages are sorted from highest to lowest baseline score and assigned a
1-based rank.

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

The CSV is generated by the notebook and is not intended to be committed
to Git.

In [28]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_cols = [
    "rank",
    "baseline_score",
    "action",
    "reason_code",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ctr_gap"
]

ranked[output_cols].to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: work/outputs/baseline_action_score.csv


In [29]:
check = pd.read_csv(output_path)

print("Shape:", check.shape)
print(check.head())

Shape: (30000, 8)
   rank  baseline_score          action  reason_code  impressions_90d   ctr  \
0     1        0.972473  REFRESH_REVIEW  HIGH_VOLUME           208678  0.00   
1     2        0.944387  REFRESH_REVIEW  HIGH_VOLUME           140079  0.01   
2     3        0.944253  REFRESH_REVIEW  HIGH_VOLUME           223271  0.03   
3     4        0.943187  REFRESH_REVIEW  HIGH_VOLUME           119217  0.02   
4     5        0.943147  REFRESH_REVIEW  HIGH_VOLUME           112434  0.01   

   avg_position  ctr_gap  
0           9.7     0.16  
1           7.6     0.15  
2           7.8     0.13  
3           7.0     0.14  
4           7.2     0.15  


In [30]:
top20 = ranked.head(20).copy()

top20[[
    "rank",
    "action",
    "reason_code",
    "baseline_score",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ctr_gap"
]]

,rank,action,reason_code,baseline_score,impressions_90d,ctr,avg_position,ctr_gap
0,1,REFRESH_REVIEW,HIGH_VOLUME,0.972473,208678,0.00,9.7,0.16
1,2,REFRESH_REVIEW,HIGH_VOLUME,0.944387,140079,0.01,7.6,0.15
2,3,REFRESH_REVIEW,HIGH_VOLUME,0.944253,223271,0.03,7.8,0.13
3,4,REFRESH_REVIEW,HIGH_VOLUME,0.943187,119217,0.02,7.0,0.14
4,5,REFRESH_REVIEW,HIGH_VOLUME,0.943147,112434,0.01,7.2,0.15
5,6,REFRESH_REVIEW,HIGH_VOLUME,0.943073,134055,0.03,7.5,0.13
6,7,REFRESH_REVIEW,HIGH_VOLUME,0.942433,123469,0.03,8.0,0.13
7,8,REFRESH_REVIEW,HIGH_VOLUME,0.942173,22456,0.00,6.6,0.16
8,9,REFRESH_REVIEW,HIGH_VOLUME,0.942140,295097,0.05,7.3,0.11
9,10,REFRESH_REVIEW,HIGH_VOLUME,0.941253,99013,0.03,6.4,0.13


## 3. Top-20 Review

The baseline ranked all pages by the hand-written baseline action score.
I reviewed the top 20 pages rather than treating the ranking as automatically
correct.

For each of the top 20 rows, the review considers three things:

1. **Action** — the action assigned by the baseline rule.
2. **Why it is ranked highly** — the signals that contributed to its score,
   mainly search volume and the CTR opportunity signal.
3. **What would make it wrong** — a possible real-world explanation that
   could make the recommended action inappropriate.

The purpose of this review is to identify weaknesses in the rule rather than
assume that every high-scoring page is a valid refresh opportunity.

### Top-20 Review Criteria

A high ranking is more convincing when a page has meaningful search
visibility and evidence of a CTR opportunity relative to its search position.

However, the baseline can still make mistakes. For example:

- High impressions do not necessarily mean that content is outdated.
- Low CTR can be caused by SERP features, search intent, title/meta
  presentation, or other factors rather than content quality.
- A page with very high search volume may already be performing reasonably
  well.
- A low CTR for a poorly ranking page may simply be a consequence of its
  position rather than evidence that the content needs refreshing.

Therefore, each of the following 20 recommendations is treated as a
hypothesis for review rather than a guaranteed refresh decision.

### Individual Top-20 Review

For each ranked page, I record the assigned action, the main reason for its
high ranking, and the evidence that could make the recommendation wrong.

| Rank | Action | Why it's there | What would make it wrong |
|---:|---|---|---|
| 1 | REFRESH_REVIEW | High search volume gives the page strong opportunity value. | The page may already be current and the low CTR may be caused by SERP presentation rather than content. |
| 2 | REFRESH_REVIEW | High search visibility contributes strongly to the baseline score. | High impressions alone do not prove that a refresh is needed. |
| 3 | REFRESH_REVIEW | Strong volume signal combined with the baseline opportunity score. | The page may already satisfy search intent despite its CTR. |
| 4 | REFRESH_REVIEW | High impressions make this a potentially valuable page to review. | The CTR may be explained by search features or query intent. |
| 5 | REFRESH_REVIEW | Search volume gives the page a high priority in the baseline. | The content may already be up to date and performing adequately. |
| 6 | REFRESH_REVIEW | High visibility increases the potential value of improvement. | The observed CTR may not be caused by content quality. |
| 7 | REFRESH_REVIEW | Strong search-volume contribution to the baseline score. | The page could be ranking for queries where low CTR is expected. |
| 8 | REFRESH_REVIEW | High impressions and the combined baseline signals produce a high score. | A manual review could show no meaningful content-refresh opportunity. |
| 9 | REFRESH_REVIEW | High search visibility makes this page worth investigating. | SERP features or intent mismatch could explain its CTR. |
| 10 | REFRESH_REVIEW | The page receives substantial search exposure. | The page may already have effective content and metadata. |
| 11 | REFRESH_REVIEW | Its search volume contributes to a high baseline score. | The low CTR may be unrelated to outdated content. |
| 12 | REFRESH_REVIEW | Strong volume signal makes the page potentially valuable to improve. | Search intent may make the observed CTR reasonable. |
| 13 | REFRESH_REVIEW | High impressions contribute to its priority ranking. | A content refresh may not improve CTR if the underlying issue is ranking. |
| 14 | REFRESH_REVIEW | The baseline identifies meaningful search visibility. | The page may already meet user intent despite the CTR signal. |
| 15 | REFRESH_REVIEW | High search exposure makes the page a useful review candidate. | High impressions alone are insufficient evidence for refreshing content. |
| 16 | REFRESH_REVIEW | The combined signals place the page among the highest priorities. | The CTR issue could be caused by SERP presentation rather than content. |
| 17 | REFRESH_REVIEW | Search volume contributes substantially to its score. | The page could already be accurate and current. |
| 18 | REFRESH_REVIEW | Its search visibility makes potential improvement valuable. | The low CTR may be normal for the queries and position involved. |
| 19 | REFRESH_REVIEW | High impressions contribute to a high baseline score. | Manual inspection may show that the content itself is not the problem. |
| 20 | REFRESH_REVIEW | The page remains within the highest-scoring baseline candidates. | The page may have insufficient evidence for a true refresh opportunity. |

### Review Observation

An important limitation appears in the top-20 queue: all 20 pages receive the
`HIGH_VOLUME` reason code. This indicates that the baseline is being driven
strongly by the search-volume component.

This is a useful weakness to document. High impressions identify pages with
large potential reach, but they do not by themselves establish that content
refreshing is the correct action. The Week-5 model should therefore test
whether additional signals can distinguish genuine refresh opportunities from
pages that simply have high search visibility.

In [32]:
weak_candidates = (
    ranked[
        (ranked["action"] == "REFRESH_REVIEW") &
        (ranked["reason_code"] == "HIGH_VOLUME")
    ]
    .sort_values(
        ["ctr_gap", "impressions_90d"],
        ascending=[True, False]
    )
    .head(10)
)

weak_candidates[[
    "rank",
    "baseline_score",
    "action",
    "reason_code",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ctr_gap"
]]

,rank,baseline_score,action,reason_code,impressions_90d,ctr,avg_position,ctr_gap
2904,2905,0.71370,REFRESH_REVIEW,HIGH_VOLUME,28192,4.78,9.1,-4.62
2057,2058,0.72956,REFRESH_REVIEW,HIGH_VOLUME,62927,3.40,7.2,-3.24
6717,6718,0.64591,REFRESH_REVIEW,HIGH_VOLUME,7485,2.62,16.2,-2.52
3442,3443,0.70404,REFRESH_REVIEW,HIGH_VOLUME,21272,2.45,12.6,-2.35
3019,3020,0.71150,REFRESH_REVIEW,HIGH_VOLUME,26287,2.43,13.2,-2.33
5229,5230,0.67246,REFRESH_REVIEW,HIGH_VOLUME,11202,2.30,3.0,-2.30
4107,4108,0.69234,REFRESH_REVIEW,HIGH_VOLUME,16281,2.33,13.5,-2.23
2107,2108,0.72870,REFRESH_REVIEW,HIGH_VOLUME,57078,2.32,5.1,-2.16
3043,3044,0.71110,REFRESH_REVIEW,HIGH_VOLUME,25978,2.27,3.7,-2.11
4870,4871,0.67877,REFRESH_REVIEW,HIGH_VOLUME,12495,2.26,4.9,-2.10


## Weak Picks

The baseline can produce false-positive refresh recommendations because
search volume receives 60% of the final score. To examine this weakness, I
looked for high-volume pages assigned `REFRESH_REVIEW` that had relatively
small or negative CTR gaps.

A negative `ctr_gap` means that the observed CTR is higher than the typical
CTR for the page's position bucket. Therefore, these pages have strong CTR
performance despite being selected by the baseline.

### Weak Pick 1 — Rank 2905

This page has 28,192 impressions and a CTR of 4.78 at an average position of
9.1. Its CTR gap is -4.62, meaning its observed CTR is substantially higher
than the expected CTR for its position bucket.

The baseline still assigns `REFRESH_REVIEW` because its high search volume
contributes strongly to the baseline score. This could be a false positive:
the page already has strong CTR performance, so high impressions alone do
not provide enough evidence that the content needs refreshing.

### Weak Pick 2 — Rank 2069

This page has 62,927 impressions and a CTR of 3.40 at an average position of
7.2. Its CTR gap is -3.24, showing that the observed CTR is substantially
above the expected level for its position bucket.

The page is prioritized mainly because of its high search visibility. It may
therefore be a weak refresh recommendation if the content is already current
and satisfying search intent.

### Weak Pick 3 — Rank 6718

This page has 7,485 impressions and a CTR of 2.62 at an average position of
16.2. Its CTR gap is -2.52, indicating stronger-than-expected CTR performance
for its position bucket.

Although the baseline assigns `REFRESH_REVIEW`, the negative CTR gap provides
evidence against a CTR-based refresh opportunity. This shows that the
volume-heavy baseline can still prioritize pages that are performing well on
CTR.

### Weak-Pick Conclusion

These examples reveal a limitation of the baseline rule. High search volume
is useful for identifying pages with greater potential reach, but it does not
prove that a page needs a content refresh. A stronger future model should
distinguish between high-volume pages that are genuinely underperforming and
high-volume pages that are already performing well.

## Leakage Check

The baseline uses only current-snapshot information:
- impressions_90d
- ctr
- avg_position
- derived position-bucket CTR gap

No future-window performance, future outcomes, or label-derived inputs
are used in the baseline score.

In [33]:
print("=== SELF CHECK ===")

print("Dataset shape:", df.shape)
print("Ranked rows:", len(ranked))

print("\nMissing baseline scores:",
      ranked["baseline_score"].isna().sum())

print("Missing actions:",
      ranked["action"].isna().sum())

print("Missing reason codes:",
      ranked["reason_code"].isna().sum())

print("\nAction counts:")
print(ranked["action"].value_counts())

print("\nReason counts:")
print(ranked["reason_code"].value_counts())

print("\nOutput exists:",
      output_path.exists())

=== SELF CHECK ===
Dataset shape: (30000, 53)
Ranked rows: 30000

Missing baseline scores: 0
Missing actions: 0
Missing reason codes: 0

Action counts:
action
MONITOR           22500
REFRESH_REVIEW     7500
Name: count, dtype: int64

Reason counts:
reason_code
HIGH_VOLUME    16491
CTR_GAP        13509
Name: count, dtype: int64

Output exists: True


In [34]:
assert ranked["baseline_score"].notna().all()
assert ranked["action"].notna().all()
assert ranked["reason_code"].notna().all()
assert ranked["rank"].is_monotonic_increasing
assert output_path.exists()

print("\nSELF-CHECK PASSED")


SELF-CHECK PASSED
